## 1. Implementation of the FFT Algorithm
We implement the recursive Cooley-Tukey algorithm. The inputs must be a power of 2. We use vectorized Numpy operations where possible for speed. This custom implementation serves as the mathematical foundation for our advanced spectral analysis framework.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import requests
from PIL import Image
import io

# Set plot aesthetics for scientific visualization
plt.rcParams['figure.figsize'] = [14, 10]
plt.rcParams['image.cmap'] = 'magma'  # Excellent for fluorescence microscopy

class FourierEngine:
    """
    A scientifically rigorous implementation of 1D and 2D FFT algorithms.
    Implements the recursive Cooley-Tukey algorithm.
    """

    @staticmethod
    def fft1d(x):
        """Recursive 1D FFT Implementation using Cooley-Tukey."""
        x = np.asarray(x, dtype=complex)
        N = x.shape[0]

        if N <= 1:
            return x

        if N % 2 != 0:
            raise ValueError("Size must be a power of 2 for this Cooley-Tukey implementation")

        even = FourierEngine.fft1d(x[0::2])
        odd = FourierEngine.fft1d(x[1::2])

        k = np.arange(N // 2)
        T = np.exp(-2j * np.pi * k / N) * odd

        return np.concatenate([even + T, even - T])

    @staticmethod
    def ifft1d(X):
        """Inverse 1D FFT Implementation using the complex conjugate property."""
        X = np.asarray(X, dtype=complex)
        N = X.shape[0]
        res = FourierEngine.fft1d(np.conj(X))
        return np.conj(res) / N

    @classmethod
    def fft2d(cls, image):
        """Compute 2D FFT using separability (row-column decomposition)."""
        image = np.asarray(image, dtype=complex)
        rows, cols = image.shape

        row_transformed = np.zeros_like(image, dtype=complex)
        for i in range(rows):
            row_transformed[i, :] = cls.fft1d(image[i, :])

        result = np.zeros_like(image, dtype=complex)
        for j in range(cols):
            result[:, j] = cls.fft1d(row_transformed[:, j])

        return result

    @classmethod
    def ifft2d(cls, spectrum):
        """Compute 2D IFFT using separability."""
        spectrum = np.asarray(spectrum, dtype=complex)
        rows, cols = spectrum.shape

        row_transformed = np.zeros_like(spectrum, dtype=complex)
        for i in range(rows):
            row_transformed[i, :] = cls.ifft1d(spectrum[i, :])

        result = np.zeros_like(spectrum, dtype=complex)
        for j in range(cols):
            result[:, j] = cls.ifft1d(row_transformed[:, j])

        return result.real

    @staticmethod
    def fft_shift(spectrum):
        """Shifts the zero-frequency component to the center of the spectrum."""
        rows, cols = spectrum.shape
        return np.roll(spectrum, (rows // 2, cols // 2), axis=(0, 1))

    @staticmethod
    def ifft_shift(spectrum):
        """Inverse of fft_shift."""
        rows, cols = spectrum.shape
        return np.roll(spectrum, (-rows // 2, -cols // 2), axis=(0, 1))


## 2. Advanced Filter Design
Based on the proposed methodology, we define and compare multiple families of Fourier-domain filters to handle different artefact classes. To mitigate "ringing" artifacts associated with the **Ideal Low-Pass Filter**, we implement a **Gaussian Low-Pass Filter** and a **Butterworth Low-Pass Filter**, which offer smoother transitions. Furthermore, a **Notch-Reject Filter** is provided to target localized periodic artefacts without degrading the overall signal.

**Filter Families:**
1. **Ideal**: Compact support with a hard cut-off. High risk of ringing.
2. **Gaussian**: Smooth, non-negative response. Suppresses noise with minimal ringing.
3. **Butterworth**: Tunable roll-off controlled by an order parameter $n$, balancing selectivity and ringing.
4. **Notch-Reject**: Selective removal of symmetric spectral peaks.


In [ ]:
from abc import ABC, abstractmethod

class BaseFrequencyFilter(ABC):
    """Abstract base class for robust OOP filter implementation."""

    @abstractmethod
    def compute_mask(self, shape):
        pass

    def apply(self, shifted_spectrum):
        mask = self.compute_mask(shifted_spectrum.shape)
        return shifted_spectrum * mask, mask

class IdealLowPassFilter(BaseFrequencyFilter):
    """Ideal low-pass filter with a hard cut-off. Prone to ringing."""
    def __init__(self, cutoff_frequency):
        self.D0 = cutoff_frequency

    def compute_mask(self, shape):
        rows, cols = shape
        crow, ccol = rows // 2, cols // 2
        y, x = np.ogrid[:rows, :cols]
        D = np.sqrt((x - ccol)**2 + (y - crow)**2)
        mask = (D <= self.D0).astype(float)
        return mask

class GaussianLowPassFilter(BaseFrequencyFilter):
    """Gaussian low pass filter prevents ringing artifacts through smooth attenuation."""
    def __init__(self, cutoff_frequency):
        self.D0 = cutoff_frequency

    def compute_mask(self, shape):
        rows, cols = shape
        crow, ccol = rows // 2, cols // 2
        y, x = np.ogrid[:rows, :cols]
        D2 = (x - ccol)**2 + (y - crow)**2
        mask = np.exp(-D2 / (2 * self.D0**2))
        return mask

class ButterworthLowPassFilter(BaseFrequencyFilter):
    """Butterworth filter provides a tunable roll-off via the order parameter."""
    def __init__(self, cutoff_frequency, order=2):
        self.D0 = cutoff_frequency
        self.n = order

    def compute_mask(self, shape):
        rows, cols = shape
        crow, ccol = rows // 2, cols // 2
        y, x = np.ogrid[:rows, :cols]
        D = np.sqrt((x - ccol)**2 + (y - crow)**2)
        # Add small epsilon to prevent division by zero at DC
        D[crow, ccol] = 1e-5
        mask = 1 / (1 + (D / self.D0)**(2 * self.n))
        mask[crow, ccol] = 1.0  # DC component passes unaltered
        return mask

class NotchRejectFilter(BaseFrequencyFilter):
    """Notch filter for removing periodic/structured artifacts."""
    def __init__(self, notch_centers, radius):
        self.notch_centers = notch_centers
        self.D0 = radius

    def compute_mask(self, shape):
        rows, cols = shape
        mask = np.ones(shape)
        y, x = np.ogrid[:rows, :cols]
        for (r, c) in self.notch_centers:
            D = np.sqrt((x - c)**2 + (y - r)**2)
            mask[D <= self.D0] = 0.0
        return mask


## 3. Data Retrieval and Apodization
We fetch a real fluorescence microscopy image from the Image Data Resource (IDR). 
Before computing the FFT, it is critical to address spectral leakage caused by finite image boundaries. We implement an **Apodization (Windowing)** step using a 2D Hann window, which tapers the edges of the image to zero, enforcing periodic boundary conditions.


In [ ]:
def apply_hann_window(image):
    """Applies a 2D Hann window to reduce spectral leakage."""
    rows, cols = image.shape
    row_window = np.hanning(rows)
    col_window = np.hanning(cols)
    window_2d = np.outer(row_window, col_window)
    return image * window_2d

def fetch_idr_image(image_id="10502672", target_size=(256, 256)):
    url = f"https://idr.openmicroscopy.org/webgateway/render_thumbnail/{image_id}/?size=512"
    print(f"Fetching IDR Image from: {url}")
    response = requests.get(url)
    if response.status_code == 200:
        img = Image.open(io.BytesIO(response.content)).convert('L')
        img_resized = img.resize(target_size, Image.Resampling.LANCZOS)
        return np.array(img_resized)
    else:
        raise Exception(f"Failed to fetch image: HTTP {response.status_code}")

try:
    original_image = fetch_idr_image("10502672")
    print("Microscopy image successfully loaded.")
except Exception as e:
    print(e)
    # Synthetic fallback
    original_image = np.zeros((256, 256))
    original_image[100:150, 100:150] = 255

# Apply apodization to reduce edge artifacts in the frequency domain
apodized_image = apply_hann_window(original_image)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(original_image, cmap='magma')
axes[0].set_title("Original Image")
axes[0].axis('off')
axes[1].imshow(apodized_image, cmap='magma')
axes[1].set_title("Apodized Image (Hann Window)")
axes[1].axis('off')
plt.show()


## 4. Spectral Analysis and Cut-off Selection
To systematically select cut-off frequencies, we compute the **Radial Power Spectrum**. This provides a 1D representation of energy distribution across frequencies, making it easier to identify the point where signal transitions into background noise.


In [ ]:
def compute_radial_profile(spectrum):
    """Computes the 1D radial power spectrum."""
    magnitude_sq = np.abs(spectrum)**2
    rows, cols = spectrum.shape
    crow, ccol = rows // 2, cols // 2
    y, x = np.ogrid[:rows, :cols]
    r = np.sqrt((x - ccol)**2 + (y - crow)**2).astype(int)
    
    tbin = np.bincount(r.ravel(), magnitude_sq.ravel())
    nr = np.bincount(r.ravel())
    radialprofile = tbin / nr
    return radialprofile

# Transform
print("Computing 2D FFT...")
spectrum = FourierEngine.fft2d(apodized_image)
shifted_spectrum = FourierEngine.fft_shift(spectrum)

# Radial Profile
radial_profile = compute_radial_profile(shifted_spectrum)

plt.figure(figsize=(10, 6))
plt.plot(radial_profile, color='purple', linewidth=2)
plt.yscale('log')
plt.title("Radial Power Spectrum (Log Scale)", fontsize=14)
plt.xlabel("Spatial Frequency (pixels from center)", fontsize=12)
plt.ylabel("Power", fontsize=12)
plt.grid(True, alpha=0.3)
plt.axvline(x=40, color='r', linestyle='--', label='Selected Cut-off ($D_0=40$)')
plt.legend(fontsize=12)
plt.show()


## 5. Filter Comparison and Quantitative Evaluation
We evaluate the different filter families on a synthetically degraded version of our image to assess denoising capabilities and the spatial consequences (e.g., ringing) of different spectral cut-offs. We use Peak Signal-to-Noise Ratio (PSNR) as a quantitative metric.


In [ ]:
def calculate_psnr(img1, img2):
    """Calculates Peak Signal-to-Noise Ratio."""
    mse = np.mean((img1 - img2) ** 2)
    if mse == 0:
        return float('inf')
    max_pixel = 255.0
    return 20 * np.log10(max_pixel / np.sqrt(mse))

# Synthetically add stochastic noise to test filter robustness
np.random.seed(42)
noisy_image = apodized_image + np.random.normal(0, 25, apodized_image.shape)
noisy_image = np.clip(noisy_image, 0, 255)

# FFT of noisy image
spectrum_noisy = FourierEngine.fft2d(noisy_image)
shifted_spectrum_noisy = FourierEngine.fft_shift(spectrum_noisy)

# Define filters to compare
cutoff = 40
filters = {
    "Ideal": IdealLowPassFilter(cutoff_frequency=cutoff),
    "Gaussian": GaussianLowPassFilter(cutoff_frequency=cutoff),
    "Butterworth (n=2)": ButterworthLowPassFilter(cutoff_frequency=cutoff, order=2)
}

fig, axes = plt.subplots(3, len(filters) + 1, figsize=(22, 16))

# Plot Noisy Input
axes[0, 0].imshow(noisy_image, cmap='magma')
axes[0, 0].set_title(f"Noisy Input (PSNR: {calculate_psnr(apodized_image, noisy_image):.2f} dB)", fontsize=14)
axes[0, 0].axis('off')

axes[1, 0].imshow(np.log(1 + np.abs(shifted_spectrum_noisy)), cmap='magma')
axes[1, 0].set_title("Noisy Spectrum", fontsize=14)
axes[1, 0].axis('off')

axes[2, 0].axis('off')

# Apply filters and plot
for i, (name, filter_obj) in enumerate(filters.items()):
    col = i + 1
    
    # Apply mask
    filtered_spectrum, mask = filter_obj.apply(shifted_spectrum_noisy)
    
    # Inverse FFT
    inverse_shifted = FourierEngine.ifft_shift(filtered_spectrum)
    restored_image = FourierEngine.ifft2d(inverse_shifted)
    restored_image = np.clip(restored_image, 0, 255)
    
    psnr_val = calculate_psnr(apodized_image, restored_image)
    
    # Restored Image
    axes[0, col].imshow(restored_image, cmap='magma')
    axes[0, col].set_title(f"Restored: {name}\n(PSNR: {psnr_val:.2f} dB)", fontsize=14)
    axes[0, col].axis('off')
    
    # Filtered Spectrum
    axes[1, col].imshow(np.log(1 + np.abs(filtered_spectrum)), cmap='magma')
    axes[1, col].set_title(f"{name} Spectrum", fontsize=14)
    axes[1, col].axis('off')
    
    # Filter Mask
    axes[2, col].imshow(mask, cmap='viridis')
    axes[2, col].set_title(f"{name} Mask", fontsize=14)
    axes[2, col].axis('off')

plt.tight_layout()
plt.show()


## 6. Periodic Artefact Removal with Notch Filters
Structured, periodic artefacts (like grid or scanner lines) appear as distinct localized peaks in the frequency domain. A **Notch-Reject Filter** effectively eliminates these peaks without degrading the global image resolution, which is highly beneficial compared to a heavy global low-pass filter.


In [ ]:
# Create a synthetic periodic stripe artefact
rows, cols = apodized_image.shape
y, x = np.ogrid[:rows, :cols]
# Frequency of the artefact
freq_y, freq_x = 35, 25
stripe_artefact = 30 * np.sin(2 * np.pi * (freq_y * y / rows + freq_x * x / cols))

image_with_stripes = apodized_image + stripe_artefact
image_with_stripes = np.clip(image_with_stripes, 0, 255)

# FFT
spectrum_stripes = FourierEngine.fft2d(image_with_stripes)
shifted_spectrum_stripes = FourierEngine.fft_shift(spectrum_stripes)

# The peaks should appear at (center_y ± freq_y, center_x ± freq_x)
crow, ccol = rows // 2, cols // 2
notch_centers = [
    (crow + freq_y, ccol + freq_x),
    (crow - freq_y, ccol - freq_x)
]

# Apply Notch Filter
notch_filter = NotchRejectFilter(notch_centers=notch_centers, radius=5)
filtered_spectrum_notch, notch_mask = notch_filter.apply(shifted_spectrum_stripes)

# Reconstruct
inverse_shifted_notch = FourierEngine.ifft_shift(filtered_spectrum_notch)
restored_from_stripes = FourierEngine.ifft2d(inverse_shifted_notch)
restored_from_stripes = np.clip(restored_from_stripes, 0, 255)

fig, axes = plt.subplots(1, 4, figsize=(22, 6))

axes[0].imshow(image_with_stripes, cmap='magma')
axes[0].set_title("Image with Stripe Artefact", fontsize=14)
axes[0].axis('off')

axes[1].imshow(np.log(1 + np.abs(shifted_spectrum_stripes)), cmap='magma')
axes[1].set_title("Spectrum (Note Localized Peaks)", fontsize=14)
axes[1].axis('off')

axes[2].imshow(notch_mask, cmap='viridis')
axes[2].set_title("Notch-Reject Filter Mask", fontsize=14)
axes[2].axis('off')

axes[3].imshow(restored_from_stripes, cmap='magma')
axes[3].set_title("Restored Image", fontsize=14)
axes[3].axis('off')

plt.tight_layout()
plt.show()
